# 🏃 Batch RL with HalfCheetah

In [1]:
# Install MuJoCo rendering dependencies for Colab
!apt-get update -qq
!apt-get install -y -qq xvfb ffmpeg > /dev/null 2>&1

print("✓ System dependencies installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✓ System dependencies installed


In [2]:
# Install Python packages
# Using Minari for D4RL-compatible datasets (works with Python 3.12)
!pip install -q gymnasium[mujoco] pyvirtualdisplay minari
!pip install -q d3rlpy

print("✓ Python packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 30.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 17.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the followi

##Setup Virtual Display


In [3]:
from pyvirtualdisplay import Display

# Start virtual display for rendering
display = Display(visible=0, size=(400, 300))
display.start()

print("✓ Virtual display ready")

✓ Virtual display ready


## Step 3: Loading the Dataset from Minari


In [4]:
import minari
import d3rlpy
import numpy as np

# Use the correct Minari dataset name
dataset_name = "mujoco/halfcheetah/medium-v0"

print(f"Downloading {dataset_name}... (this may take a minute)")
minari_dataset = minari.load_dataset(dataset_name, download=True)

# Convert to d3rlpy format
observations = []
actions = []
rewards = []
terminals = []

print("Converting dataset to d3rlpy format...")
for episode in minari_dataset:
    observations.append(episode.observations[:-1])  # All but last
    actions.append(episode.actions)
    rewards.append(episode.rewards)

    # Create terminal flags
    term = np.zeros(len(episode.rewards), dtype=bool)
    term[-1] = True
    terminals.append(term)

# Flatten all episodes
observations = np.vstack(observations)
actions = np.vstack(actions)
rewards = np.concatenate(rewards)
terminals = np.concatenate(terminals)

# Create d3rlpy dataset
dataset = d3rlpy.dataset.MDPDataset(
    observations=observations,
    actions=actions,
    rewards=rewards,
    terminals=terminals
)

print(f"\n✓ Dataset loaded: {len(observations)} transitions")
print(f"  Observation shape: {observations.shape}")
print(f"  Action shape: {actions.shape}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


namespace_metadata.json: 0.00B [00:00, ?B/s]

metadata.json: 0.00B [00:00, ?B/s]

namespace_metadata.json: 0.00B [00:00, ?B/s]

...

namespace_metadata.json:   0%|          | 0.00/267 [00:00<?, ?B/s]

namespace_metadata.json: 0.00B [00:00, ?B/s]

namespace_metadata.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]


Dataset mujoco/halfcheetah/medium-v0 downloaded to /root/.minari/datasets/mujoco/halfcheetah/medium-v0
Converting dataset to d3rlpy format...
2026-04-08 20:42.29 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(6,)]) observation_signature=Signature(dtype=[dtype('float64')], shape=[(17,)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2026-04-08 20:42.29 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2026-04-08 20:42.29 [info     ] Action size has been automatically determined. action_size=6

✓ Dataset loaded: 1000000 transitions
  Observation shape: (1000000, 17)
  Action shape: (1000000, 6)


##Training the Agent

In [9]:
import torch

# Check if GPU is available
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"🚀 Training on: {device.upper()}")

# Create CQL agent with better hyperparameters
cql = d3rlpy.algos.CQLConfig(
    actor_learning_rate=3e-4,
    critic_learning_rate=3e-4,
    batch_size=256,
).create(device=device)

# Train for MUCH longer (this is the key!)
# 100k-500k steps for decent results, 1M+ for best results
cql.fit(
    dataset,
    n_steps=100000,  # Changed from 10000 to 100000 (10x more)
    n_steps_per_epoch=10000,  # Log every 10k steps
    show_progress=True
)

print("\n✓ Training complete!")

🚀 Training on: CUDA:0
2026-04-08 20:55.16 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float64')], shape=[(17,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(6,)]), reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=6)
2026-04-08 20:55.16 [debug    ] Building models...            
2026-04-08 20:55.16 [debug    ] Models have been built.       
2026-04-08 20:55.16 [info     ] Directory is created at d3rlpy_logs/CQL_20260408205516
2026-04-08 20:55.16 [info     ] Parameters                     params={'observation_shape': [17], 'action_size': 6, 'config': {'type': 'cql', 'params': {'batch_size': 256, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'actor_learning_rate': 0.0003, 'critic_learning

Epoch 1/10:   0%|          | 0/10000 [00:00<?, ?it/s]

2026-04-08 20:59.32 [info     ] CQL_20260408205516: epoch=1 step=10000 epoch=1 metrics={'time_sample_batch': 0.003823779845237732, 'time_algorithm_update': 0.021308030915260314, 'critic_loss': 212.87286922912597, 'conservative_loss': -50.949572743797304, 'alpha': 0.6257171149820089, 'actor_loss': -285.9431380899429, 'temp': 0.7303980511486531, 'temp_loss': 2.28188456206657, 'time_step': 0.025431738543510437} step=10000
2026-04-08 20:59.32 [info     ] Model parameters are saved to d3rlpy_logs/CQL_20260408205516/model_10000.d3


Epoch 2/10:   0%|          | 0/10000 [00:00<?, ?it/s]

2026-04-08 21:03.50 [info     ] CQL_20260408205516: epoch=2 step=20000 epoch=2 metrics={'time_sample_batch': 0.0038759654998779297, 'time_algorithm_update': 0.0214891432762146, 'critic_loss': 1918.473771995163, 'conservative_loss': -5.190021661812626, 'alpha': 0.3076949925094843, 'actor_loss': -893.7799035644531, 'temp': 1.1450620592236518, 'temp_loss': -0.6096379975086078, 'time_step': 0.025672536230087282} step=20000
2026-04-08 21:03.50 [info     ] Model parameters are saved to d3rlpy_logs/CQL_20260408205516/model_20000.d3


...

2026-04-08 21:29.52 [info     ] CQL_20260408205516: epoch=8 step=80000 epoch=8 metrics={'time_sample_batch': 0.0038208844900131226, 'time_algorithm_update': 0.02130312819480896, 'critic_loss': 5555.582544774627, 'conservative_loss': -0.4181674451492727, 'alpha': 0.012179866195656358, 'actor_loss': -1691.4292646728516, 'temp': 0.8685076077818871, 'temp_loss': 0.010727817248553038, 'time_step': 0.02542738370895386} step=80000
2026-04-08 21:29.52 [info     ] Model parameters are saved to d3rlpy_logs/CQL_20260408205516/model_80000.d3


Epoch 9/10:   0%|          | 0/10000 [00:00<?, ?it/s]

2026-04-08 21:34.09 [info     ] CQL_20260408205516: epoch=9 step=90000 epoch=9 metrics={'time_sample_batch': 0.0038642664194107057, 'time_algorithm_update': 0.021447648882865905, 'critic_loss': 5854.3265953155515, 'conservative_loss': -0.15591483065830544, 'alpha': 0.005055676984903403, 'actor_loss': -1721.8634716064453, 'temp': 0.821133686631918, 'temp_loss': 0.026325720359105617, 'time_step': 0.025620507836341858} step=90000
2026-04-08 21:34.09 [info     ] Model parameters are saved to d3rlpy_logs/CQL_20260408205516/model_90000.d3


Epoch 10/10:   0%|          | 0/10000 [00:00<?, ?it/s]

2026-04-08 21:38.29 [info     ] CQL_20260408205516: epoch=10 step=100000 epoch=10 metrics={'time_sample_batch': 0.0038968002796173095, 'time_algorithm_update': 0.021650667667388917, 'critic_loss': 6009.232740571594, 'conservative_loss': -0.06102379542675335, 'alpha': 0.0020953753025620245, 'actor_loss': -1739.7248869384766, 'temp': 0.7961592822492123, 'temp_loss': 0.0005123878696933388, 'time_step': 0.025864090037345886} step=100000
2026-04-08 21:38.29 [info     ] Model parameters are saved to d3rlpy_logs/CQL_20260408205516/model_100000.d3

✓ Training complete!


## Model saved as cheetah_agent.d3


In [10]:
# Save trained agent
cql.save("cheetah_agent.d3")

print("✓ Model saved as 'cheetah_agent.d3'")

✓ Model saved as 'cheetah_agent.d3'


##Visual Representation of Cheetah Run !


In [11]:
import gymnasium as gym
import numpy as np
from IPython.display import Video
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Create environment with rendering
env = gym.make("HalfCheetah-v4", render_mode="rgb_array")

# Record one episode
frames = []
observation, info = env.reset()
total_reward = 0

print("Recording episode...")
for step in range(1000):  # Max 1000 steps
    # Get action from trained agent
    action = cql.predict(observation.reshape(1, -1))[0]

    # Take action in environment
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    # Save frame
    frames.append(env.render())

    if terminated or truncated:
        print(f"Episode finished after {step + 1} steps")
        break

env.close()
print(f"✓ Total reward: {total_reward:.2f}")
print(f"✓ Recorded {len(frames)} frames")

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:517: DeprecationWarning: WARN: The environment HalfCheetah-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Recording episode...
Episode finished after 1000 steps
✓ Total reward: -472.00
✓ Recorded 1000 frames


In [12]:
# Create and display video
fig, ax = plt.subplots(figsize=(6, 4))
ax.axis('off')
img = ax.imshow(frames[0])

def animate(frame_idx):
    img.set_data(frames[frame_idx])
    return [img]

anim = animation.FuncAnimation(
    fig, animate, frames=len(frames), interval=50, blit=True
)

plt.close()  # Don't show the plot, just the animation

# Display in notebook
HTML(anim.to_html5_video())

##Video saved as MP4

In [13]:
# Save to file (optional)
anim.save('cheetah_running.mp4', writer='ffmpeg', fps=20)
print("✓ Video saved as 'cheetah_running.mp4'")

# Also display the saved video
Video('cheetah_running.mp4', width=500)

✓ Video saved as 'cheetah_running.mp4'
